In [1]:
import sys
print(sys.executable)


/Users/mayankyadav/iot-ids/.venv/bin/python


In [2]:
import pandas as pd
df = pd.read_csv("../../data/train_test_network.csv")
print(df.shape)
print(df["label"].value_counts())
print(df["type"].value_counts())

(211043, 44)
label
1    161043
0     50000
Name: count, dtype: int64
type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
ransomware    20000
scanning      20000
xss           20000
mitm           1043
Name: count, dtype: int64


In [3]:
print(df.dtypes.value_counts())
print("missing:", df.isna().sum().sum(), "| duplicates:", df.duplicated().sum())
print(df.columns.tolist())

str        27
int64      16
float64     1
Name: count, dtype: int64
missing: 0 | duplicates: 20569
['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type']


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

drop_cols = ["ts", "src_ip", "dst_ip", "src_port", "label", "type"]
X = df.drop(columns=drop_cols, errors="ignore")
y_bin, y_multi = df["label"], df["type"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
print("dropped:", [c for c in drop_cols if c in df.columns])
print("categorical:", cat_cols)

X_train, X_test, yb_train, yb_test, ym_train, ym_test = train_test_split(
    X, y_bin, y_multi, test_size=0.2, stratify=y_multi, random_state=42)
X_train, X_test = X_train.copy(), X_test.copy()

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[cat_cols] = enc.fit_transform(X_train[cat_cols].astype(str))
X_test[cat_cols] = enc.transform(X_test[cat_cols].astype(str))
print(X_train.shape, X_test.shape)

/var/folders/_g/tf34ycn52fs54449r6cgv0t00000gn/T/ipykernel_84642/3032018630.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


dropped: ['src_ip', 'dst_ip', 'src_port', 'label', 'type']
categorical: ['proto', 'service', 'conn_state', 'dns_query', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice']
(168834, 39) (42209, 39)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                            class_weight="balanced", random_state=42)
rf.fit(X_train, yb_train)
pred = rf.predict(X_test)
print(classification_report(yb_test, pred, digits=4))
print(confusion_matrix(yb_test, pred))

              precision    recall  f1-score   support

           0     0.9978    0.9974    0.9976     10000
           1     0.9992    0.9993    0.9993     32209

    accuracy                         0.9989     42209
   macro avg     0.9985    0.9984    0.9984     42209
weighted avg     0.9989    0.9989    0.9989     42209

[[ 9974    26]
 [   22 32187]]


In [6]:
rf_m = RandomForestClassifier(n_estimators=200, n_jobs=-1,
class_weight="balanced", random_state=42)
rf_m.fit(X_train, ym_train)
print(classification_report(ym_test, rf_m.predict(X_test), digits=4))

              precision    recall  f1-score   support

    backdoor     0.9995    0.9998    0.9996      4000
        ddos     0.9844    0.9810    0.9827      4000
         dos     0.9847    0.9802    0.9825      4000
   injection     0.9692    0.9680    0.9686      4000
        mitm     0.6815    0.8086    0.7396       209
      normal     0.9988    0.9966    0.9977     10000
    password     0.9938    0.9938    0.9938      4000
  ransomware     0.9988    0.9990    0.9989      4000
    scanning     0.9880    0.9872    0.9876      4000
         xss     0.9707    0.9758    0.9732      4000

    accuracy                         0.9873     42209
   macro avg     0.9569    0.9690    0.9624     42209
weighted avg     0.9876    0.9873    0.9874     42209



In [7]:
print("duplicate rows in data:", df.duplicated().sum())
overlap = X_test.merge(X_train.drop_duplicates(), how="inner").shape[0]
print("test rows identical to a training row:", overlap, "of", len(X_test))

duplicate rows in data: 20569
test rows identical to a training row: 22650 of 42209


In [8]:
import pandas as pd
imp = pd.Series(rf_m.feature_importances_, index=X_train.columns)
print(imp.sort_values(ascending=False).head(15))

src_ip_bytes    0.204757
dst_port        0.143052
conn_state      0.109945
duration        0.109080
dst_ip_bytes    0.080630
src_pkts        0.071118
src_bytes       0.067610
dst_pkts        0.053014
dst_bytes       0.042079
service         0.028021
proto           0.016511
dns_RA          0.016224
dns_query       0.015423
dns_rejected    0.012014
dns_RD          0.008621
dtype: float64


In [9]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

le = LabelEncoder()
ym_tr, ym_te = le.fit_transform(ym_train), le.transform(ym_test)
xgb = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1,
                    tree_method="hist", n_jobs=-1, random_state=42)
xgb.fit(X_train, ym_tr, sample_weight=compute_sample_weight("balanced", ym_tr))
print(classification_report(ym_te, xgb.predict(X_test),
                            target_names=le.classes_, digits=4))

              precision    recall  f1-score   support

    backdoor     0.9995    0.9995    0.9995      4000
        ddos     0.9897    0.9825    0.9861      4000
         dos     0.9914    0.9805    0.9859      4000
   injection     0.9837    0.9667    0.9752      4000
        mitm     0.6327    0.8900    0.7396       209
      normal     0.9985    0.9962    0.9973     10000
    password     0.9957    0.9910    0.9934      4000
  ransomware     0.9988    0.9990    0.9989      4000
    scanning     0.9851    0.9925    0.9888      4000
         xss     0.9710    0.9875    0.9792      4000

    accuracy                         0.9890     42209
   macro avg     0.9546    0.9785    0.9644     42209
weighted avg     0.9898    0.9890    0.9893     42209



In [10]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(data, drop_extra=[]):
    Xd = data.drop(columns=drop_cols + drop_extra, errors="ignore")
    ym = data["type"]
    cats = Xd.select_dtypes(include="object").columns.tolist()
    Xtr, Xte, ytr, yte = train_test_split(
        Xd, ym, test_size=0.2, stratify=ym, random_state=42)
    Xtr, Xte = Xtr.copy(), Xte.copy()
    e = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    Xtr[cats] = e.fit_transform(Xtr[cats].astype(str))
    Xte[cats] = e.transform(Xte[cats].astype(str))
    m = RandomForestClassifier(n_estimators=200, n_jobs=-1,
    class_weight="balanced", random_state=42).fit(Xtr, ytr)
    p = m.predict(Xte)
    return round(accuracy_score(yte, p), 4), round(f1_score(yte, p, average="macro"), 4)

feat = [c for c in df.columns if c not in drop_cols]
d = df.drop_duplicates(subset=feat + ["type"]).reset_index(drop=True)
print("rows:", len(df), "->", len(d))

print("original                 :", evaluate(df))
print("deduplicated             :", evaluate(d))
print("dedup, no dst_port       :", evaluate(d, ["dst_port"]))
print("dedup, no dst_port/service:", evaluate(d, ["dst_port", "service"]))

rows: 211043 -> 105621


/var/folders/_g/tf34ycn52fs54449r6cgv0t00000gn/T/ipykernel_84642/3736300112.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cats = Xd.select_dtypes(include="object").columns.tolist()


original                 : (0.9873, 0.9624)


/var/folders/_g/tf34ycn52fs54449r6cgv0t00000gn/T/ipykernel_84642/3736300112.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cats = Xd.select_dtypes(include="object").columns.tolist()


deduplicated             : (0.9793, 0.9595)


/var/folders/_g/tf34ycn52fs54449r6cgv0t00000gn/T/ipykernel_84642/3736300112.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cats = Xd.select_dtypes(include="object").columns.tolist()


dedup, no dst_port       : (0.9774, 0.9561)


/var/folders/_g/tf34ycn52fs54449r6cgv0t00000gn/T/ipykernel_84642/3736300112.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cats = Xd.select_dtypes(include="object").columns.tolist()


dedup, no dst_port/service: (0.9772, 0.9561)


In [11]:
import joblib, os

d = df.drop_duplicates(subset=feat + ["type"]).reset_index(drop=True)
Xf = d.drop(columns=drop_cols + ["dst_port"], errors="ignore")
yf = d["type"]
cats = Xf.select_dtypes(include="object").columns.tolist()
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.2, stratify=yf, random_state=42)
Xtr, Xte = Xtr.copy(), Xte.copy()

e = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
Xtr[cats] = e.fit_transform(Xtr[cats].astype(str))
Xte[cats] = e.transform(Xte[cats].astype(str))

final = RandomForestClassifier(n_estimators=200, n_jobs=-1,
                               class_weight="balanced", random_state=42).fit(Xtr, ytr)
print(classification_report(yte, final.predict(Xte), digits=4))

os.makedirs("../../models", exist_ok=True)
joblib.dump({"model": final, "encoder": e, "cats": cats,
             "columns": Xtr.columns.tolist()}, "../../models/ids_rf.joblib")
Xte.head(3).assign(type=yte.head(3).values).to_json("../../models/sample.json", orient="records", indent=2)

/var/folders/_g/tf34ycn52fs54449r6cgv0t00000gn/T/ipykernel_84642/1784962064.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cats = Xf.select_dtypes(include="object").columns.tolist()


              precision    recall  f1-score   support

    backdoor     0.9938    0.9979    0.9959       485
        ddos     0.9785    0.9813    0.9799      2510
         dos     0.9895    0.9737    0.9815      2895
   injection     0.9846    0.9668    0.9756      3970
        mitm     0.7092    0.8558    0.7756       208
      normal     0.9965    0.9924    0.9944      4870
    password     0.9902    0.9866    0.9884      3071
  ransomware     0.9464    1.0000    0.9724       247
    scanning     0.9499    0.9643    0.9571      1121
         xss     0.9234    0.9582    0.9405      1748

    accuracy                         0.9774     21125
   macro avg     0.9462    0.9677    0.9561     21125
weighted avg     0.9783    0.9774    0.9777     21125



In [12]:
raw = Xf.loc[Xte.head(3).index].copy()
raw["true_type"] = yte.head(3).values
raw.to_json("../../models/sample.json", orient="records", indent=2)
print(raw.head(3).T)

                            48021     71575      39997
proto                         tcp       tcp        tcp
service                      http         -       http
duration                 1.335167  0.336277   1.087382
src_bytes                     300         0        244
dst_bytes                   12183         0      12183
conn_state                     SF        S0         SF
missed_bytes                    0         0          0
src_pkts                        6       120          6
src_ip_bytes                  620      4800        564
dst_pkts                       12         0         12
dst_ip_bytes                12815         0      12815
dns_query                       -         -          -
dns_qclass                      0         0          0
dns_qtype                       0         0          0
dns_rcode                       0         0          0
dns_AA                          -         -          -
dns_RD                          -         -          -
dns_RA    

In [13]:
import json
raw = Xf.loc[Xte.head(3).index].copy()
one = json.loads(raw.head(1).to_json(orient="records"))[0]
print("TRUE LABEL:", yte.head(1).values[0])
print(json.dumps(one, indent=2))

TRUE LABEL: injection
{
  "proto": "tcp",
  "service": "http",
  "duration": 1.335167,
  "src_bytes": 300,
  "dst_bytes": 12183,
  "conn_state": "SF",
  "missed_bytes": 0,
  "src_pkts": 6,
  "src_ip_bytes": 620,
  "dst_pkts": 12,
  "dst_ip_bytes": 12815,
  "dns_query": "-",
  "dns_qclass": 0,
  "dns_qtype": 0,
  "dns_rcode": 0,
  "dns_AA": "-",
  "dns_RD": "-",
  "dns_RA": "-",
  "dns_rejected": "-",
  "ssl_version": "-",
  "ssl_cipher": "-",
  "ssl_resumed": "-",
  "ssl_established": "-",
  "ssl_subject": "-",
  "ssl_issuer": "-",
  "http_trans_depth": "-",
  "http_method": "-",
  "http_uri": "-",
  "http_version": "-",
  "http_request_body_len": 0,
  "http_response_body_len": 0,
  "http_status_code": 0,
  "http_user_agent": "-",
  "http_orig_mime_types": "-",
  "http_resp_mime_types": "-",
  "weird_name": "-",
  "weird_addl": "-",
  "weird_notice": "-"
}


In [14]:
import requests, json
rows = json.loads(Xf.loc[Xte.index[:10]].to_json(orient="records"))
truth = yte.iloc[:10].tolist()
hits = 0
for r, t in zip(rows, truth):
    resp = requests.post("http://127.0.0.1:8000/predict", json=r).json()
    hits += resp["prediction"] == t
    print(f"true={t:<10} pred={resp['prediction']:<10} conf={resp['confidence']}")
print(f"{hits}/10 correct")

true=injection  pred=injection  conf=1.0
true=normal     pred=normal     conf=0.975
true=injection  pred=injection  conf=1.0
true=normal     pred=normal     conf=1.0
true=normal     pred=normal     conf=1.0
true=normal     pred=normal     conf=1.0
true=injection  pred=injection  conf=1.0
true=injection  pred=injection  conf=1.0
true=dos        pred=scanning   conf=0.59
true=ddos       pred=ddos       conf=0.905
9/10 correct
